# Mapa de influencia y desinformación — Mundial 2026
## Tarea 3, 4 y 5: Centralidad, detección de bots y visualización
**Autor:** _(completar con el nombre del integrante)_

Este notebook documenta el proceso de las Tareas 3, 4 y 5 del proyecto.
Depende de que `edges.csv` y `nodes.csv` estén en la misma carpeta, y de
que `mundial2026_analisis_red.py` esté disponible para importar las
funciones de Tarea 1 (construcción del grafo) y Tarea 2 (comunidades),
ya que Tarea 3/4/5 se construyen sobre esos resultados.

## 0. Imports y configuración

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Funciones de Tarea 1 y Tarea 2 (ya construidas en el script principal)
from mundial2026_analisis_red import (
    cargar_datos,
    construir_grafo,
    detectar_comunidades,
)

## 1. Cargar datos y reconstruir el grafo (Tarea 1 y 2)

Se reutilizan las mismas funciones del script principal para no duplicar
lógica ni arriesgarnos a que el notebook use un grafo distinto al que se
usó en la entrega final.

In [ ]:
edges_df, nodes_df = cargar_datos("edges.csv", "nodes.csv")

G = construir_grafo(edges_df)
print(f"Nodos: {G.number_of_nodes()}  |  Enlaces: {G.number_of_edges()}")

In [ ]:
particion, comunidades, modularidad = detectar_comunidades(G)
print(f"Modularidad: {modularidad:.4f}  |  Comunidades: {len(comunidades)}")

## 2. Tarea 3 — Centralidad e influencers

Se calculan tres métricas de centralidad: degree, betweenness y
eigenvector centrality.

**Nota sobre el peso en `betweenness_centrality`:** NetworkX interpreta
el parámetro `weight` como una **distancia** para el cálculo de caminos
más cortos, no como fuerza de conexión (fuente: documentación oficial de
`networkx.algorithms.centrality.betweenness_centrality`: *"Weights are
used to calculate weighted shortest paths, so they are interpreted as
distances"*). Como en nuestro caso `weight` = número de menciones (a
mayor valor, conexión más fuerte, no más "lejana"), se usa su inverso
como distancia para que las cuentas con más interacción se traten como
más cercanas. Para `eigenvector_centrality` esto no aplica: ahí el peso
sí se usa directamente como fuerza de conexión, que es el comportamiento
deseado.

In [ ]:
def calcular_centralidades(G: nx.DiGraph) -> pd.DataFrame:
    """Calcula degree, betweenness (corregida) y eigenvector centrality."""
    G_und = G.to_undirected()

    # Distancia = inverso del peso, para que "más menciones" = "más cerca"
    for _, _, data in G_und.edges(data=True):
        data["distancia"] = 1 / data["weight"] if data.get("weight", 0) else 1

    degree_c = nx.degree_centrality(G_und)
    betweenness_c = nx.betweenness_centrality(G_und, weight="distancia")
    try:
        eigenvector_c = nx.eigenvector_centrality(G_und, weight="weight", max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        eigenvector_c = nx.eigenvector_centrality_numpy(G_und, weight="weight")

    df = pd.DataFrame({"node": list(G_und.nodes())})
    df["degree_centrality"] = df["node"].map(degree_c)
    df["betweenness_centrality"] = df["node"].map(betweenness_c)
    df["eigenvector_centrality"] = df["node"].map(eigenvector_c)
    return df


centralidad_df = calcular_centralidades(G)
centralidad_df.sort_values("eigenvector_centrality", ascending=False).head(10)

### 2.1 Top 10 influencers, con justificación cualitativa

Se cruza cada cuenta con sus metadatos de `nodes.csv` (tipo, país) y la
comunidad a la que pertenece, y se justifica su influencia según qué
métrica(s) destacan (percentil 75/90 respecto al resto de la red).

In [ ]:
def reportar_centralidades(centralidad_df, nodes_df, particion, top_n=10) -> pd.DataFrame:
    nodes_idx = nodes_df.set_index("node")
    info = centralidad_df.set_index("node").join(nodes_idx, how="left")
    info["comunidad"] = info.index.map(particion)

    top = info.sort_values("eigenvector_centrality", ascending=False).head(top_n)

    resumen = []
    for nodo, fila in top.iterrows():
        justificacion = []
        if fila["degree_centrality"] >= info["degree_centrality"].quantile(0.75):
            justificacion.append("alto volumen de interacción directa")
        if fila["betweenness_centrality"] >= info["betweenness_centrality"].quantile(0.75):
            justificacion.append("actúa como puente entre comunidades")
        if fila["eigenvector_centrality"] >= info["eigenvector_centrality"].quantile(0.90):
            justificacion.append("conectada a otras cuentas muy influyentes")
        if not justificacion:
            justificacion.append("influencia moderada y distribuida")

        resumen.append({
            "node": nodo,
            "tipo": fila.get("tipo", "N/D"),
            "pais": fila.get("pais", "N/D"),
            "comunidad": fila.get("comunidad", "N/D"),
            "degree_centrality": round(fila["degree_centrality"], 4),
            "betweenness_centrality": round(fila["betweenness_centrality"], 4),
            "eigenvector_centrality": round(fila["eigenvector_centrality"], 4),
            "justificacion": "; ".join(justificacion),
        })

    return pd.DataFrame(resumen)


top_influencers_df = reportar_centralidades(centralidad_df, nodes_df, particion, top_n=10)
top_influencers_df

In [ ]:
top_influencers_df.to_csv("centralidad_output.csv", index=False)
print("Guardado: centralidad_output.csv")

## 3. Tarea 4 — Detección de bots

Detector "ajustado": lista de bots conocidos (`tipo == "bot"` en
`nodes.csv`) + reglas suaves para candidatos adicionales.

**Hallazgo importante calibrando este dataset:** con un grafo tan
pequeño y disperso como este, el coeficiente de clustering por sí solo
NO separa bien bots de cuentas reales (la gran mayoría de fans también
tiene clustering = 0), y una regla de "conecta con ≥3 comunidades" puede
marcar como sospechosas a cuentas reales que simplemente fueron
mencionadas por un bot amplificador (falso positivo: `fan_mex_10`,
`fan_usa_10`, `fan_can_10` son mencionados por `bot_fut_04`, no son bots
ellos mismos). Por eso los candidatos por regla suave se muestran
**aparte**, para revisión manual, en vez de mezclarse automáticamente
con los bots confirmados.

In [ ]:
TIPOS_WHITELIST = {"seleccion", "medio", "influencer", "hashtag"}


def detectar_bots(G: nx.DiGraph, centralidad_df, nodes_df, particion):
    nodes_idx = nodes_df.set_index("node")
    G_und = G.to_undirected()
    clustering = nx.clustering(G_und, weight="weight")
    cent_idx = centralidad_df.set_index("node")

    bots_confirmados = set(nodes_idx[nodes_idx["tipo"] == "bot"].index)

    pool = nodes_idx[
        (~nodes_idx["tipo"].isin(TIPOS_WHITELIST))
        & (~nodes_idx.index.isin(bots_confirmados))
    ].index

    umbral_degree = cent_idx.loc[cent_idx.index.isin(pool), "degree_centrality"].quantile(0.90)

    candidatos_revision = set()
    for nodo in pool:
        if nodo not in G_und:
            continue
        deg = cent_idx.loc[nodo, "degree_centrality"] if nodo in cent_idx.index else 0
        clust = clustering.get(nodo, 0)
        comunidades_vecinas = {particion[v] for v in G_und.neighbors(nodo)}
        if deg >= umbral_degree and clust == 0 and len(comunidades_vecinas) >= 3:
            candidatos_revision.add(nodo)

    filas = []
    for bot in sorted(bots_confirmados):
        comunidad = particion.get(bot, "N/D")
        menciona_a = [f"{t} (peso {G[bot][t].get('weight', 1)})" for t in G.successors(bot)] if bot in G else []
        mencionado_por = [f"{s} (peso {G[s][bot].get('weight', 1)})" for s in G.predecessors(bot)] if bot in G else []
        filas.append({
            "bot": bot,
            "comunidad": comunidad,
            "menciona_a": "; ".join(menciona_a),
            "mencionado_por": "; ".join(mencionado_por),
        })

    return bots_confirmados, candidatos_revision, pd.DataFrame(filas)


bots_confirmados, candidatos_revision, bots_detalle_df = detectar_bots(
    G, centralidad_df, nodes_df, particion
)

print(f"Bots confirmados ({len(bots_confirmados)}):", sorted(bots_confirmados))
print(f"\nCandidatos por regla suave, revisión manual ({len(candidatos_revision)}):", sorted(candidatos_revision))

In [ ]:
bots_detalle_df

In [ ]:
bots_detalle_df.to_csv("bots_detectados.csv", index=False)
print("Guardado: bots_detectados.csv")

## 4. Tarea 5 — Visualización interactiva (PyVis)

Genera `grafo_mundial2026.html`: colores por comunidad, tamaño de nodo
según eigenvector centrality (misma métrica del top de influencers en
Tarea 3, para mantener consistencia), bots en rojo, y panel lateral con
estadísticas globales + top influencers.

In [ ]:
from pyvis.network import Network


def generar_visualizacion(G, particion, centralidad_df, nodes_df, bots_confirmados,
                           modularidad, output_path="grafo_mundial2026.html"):
    G_und = G.to_undirected()
    cent_idx = centralidad_df.set_index("node")
    nodes_idx = nodes_df.set_index("node")

    net = Network(height="800px", width="100%", bgcolor="#ffffff",
                  font_color="black", notebook=False, directed=False)
    net.from_nx(G_und)

    comunidades = sorted(set(particion.values()))
    palette = plt.colormaps["tab20"].resampled(max(len(comunidades), 1))
    color_map = {com: palette(i) for i, com in enumerate(comunidades)}

    def escalar_tamano(valor):
        return 10 + (valor * 80)

    for node in net.nodes:
        node_id = str(node["id"])
        comunidad = particion.get(node_id)
        eig = cent_idx.loc[node_id, "eigenvector_centrality"] if node_id in cent_idx.index else 0
        tipo = nodes_idx.loc[node_id, "tipo"] if node_id in nodes_idx.index else "N/D"
        pais = nodes_idx.loc[node_id, "pais"] if node_id in nodes_idx.index else "N/D"

        if comunidad is not None:
            rgba = color_map[comunidad]
            hex_color = "#%02x%02x%02x" % (int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))
            node["color"] = hex_color

        node["size"] = escalar_tamano(eig)
        node["title"] = (
            f"Nodo: {node_id}<br>Tipo: {tipo} · País: {pais}<br>"
            f"Comunidad: {comunidad}<br>Eigenvector centrality: {eig:.4f}"
        )

        if node_id in bots_confirmados:
            node["color"] = "#ff0000"
            node["title"] += "<br><b>⚠ BOT DETECTADO</b>"

    net.write_html(output_path)

    top_influencers = cent_idx.sort_values("eigenvector_centrality", ascending=False).head(10)
    influencers_html = "<ul>" + "".join(
        f"<li>{n}: {r['eigenvector_centrality']:.4f}</li>" for n, r in top_influencers.iterrows()
    ) + "</ul>"

    with open(output_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    panel_html = f"""
<div id="panel-estadisticas" style="position: fixed; top: 20px; right: 20px; width: 320px;
background: #ffffff; border: 2px solid #333; padding: 15px; border-radius: 10px;
box-shadow: 0px 0px 10px rgba(0,0,0,0.3); font-family: Arial; z-index: 9999;">
<h3 style="margin-top:0;">📊 Estadísticas Globales</h3>
<p><b>Nodos:</b> {G.number_of_nodes()}</p>
<p><b>Enlaces:</b> {G.number_of_edges()}</p>
<p><b>Comunidades:</b> {len(comunidades)}</p>
<p><b>Modularidad:</b> {modularidad:.4f}</p>
<p><b>Bots detectados:</b> {len(bots_confirmados)}</p>
<hr>
<h4>🌟 Top 10 Influencers (eigenvector)</h4>
{influencers_html}
<hr>
<button onclick="mostrarSoloBots()" style="padding:8px; background:#ff0000; color:white;
border:none; border-radius:5px; cursor:pointer;">Mostrar solo bots</button>
<button onclick="mostrarTodos()" style="padding:8px; margin-top:5px; background:#333;
color:white; border:none; border-radius:5px; cursor:pointer;">Mostrar todos</button>
</div>
<script>
function mostrarSoloBots() {{
    var allNodes = nodes.get();
    allNodes.forEach(function(n) {{
        if (n.title && n.title.includes('BOT DETECTADO')) {{
            nodes.update({{id: n.id, hidden: false}});
        }} else {{
            nodes.update({{id: n.id, hidden: true}});
        }}
    }});
}}
function mostrarTodos() {{
    var allNodes = nodes.get();
    allNodes.forEach(function(n) {{ nodes.update({{id: n.id, hidden: false}}); }});
}}
</script>
</body>
"""
    html_content = html_content.replace("</body>", panel_html)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"HTML interactivo generado en: {output_path}")


generar_visualizacion(G, particion, centralidad_df, nodes_df, bots_confirmados,
                       modularidad, output_path="grafo_mundial2026.html")

## 5. Resumen de entregables generados

- `centralidad_output.csv` — Tarea 3
- `bots_detectados.csv` — Tarea 4
- `grafo_mundial2026.html` — Tarea 5

Abre `grafo_mundial2026.html` en el navegador para revisar la
visualización interactiva antes de incluir capturas en el informe final
(Tarea 6).